# Объединение датасетов и пересборка BPE-токенизатора

Ноутбук выполняет:
1. Обработку новых видео (pt16–pt21) из `Dataset_VSR/` → извлечение губ → `.pkl`
2. Сбор текстового корпуса из существующих `.pkl` + транскрипций новых видео
3. Обучение нового BPE-токенизатора на объединённом корпусе
4. Ретокенизацию всех `.pkl` (старых + новых) единым токенизатором
5. Обновление конфига модели

## Параметры

In [1]:
from pathlib import Path
import re

# ── Пути ──
# Существующий датасет (pkl с GPT-2 input_ids)
OLD_DATA_PATH     = Path("D:/Datasets/Dataset_720_proc")

# Новые видео (mp4 + csv)
NEW_RAW_PATH      = Path("D:/Datasets/Dataset_VSR")
NEW_VIDEO_PATH    = NEW_RAW_PATH / "video"
NEW_TRANSCRIPT_PATH = NEW_RAW_PATH / "transcript"

# Куда сохранить обработанные новые видео (pkl)
NEW_PROCESSED_PATH = Path("D:/Datasets/Dataset_VSR_proc")

# Итоговый объединённый датасет (BPE токенизация)
MERGED_DATA_PATH  = Path("D:/Datasets/Dataset_merged_bpe")

# Токенизатор и конфиг
TOKENIZER_PATH    = Path("configs/bpe_tokenizer_merged")
DST_CONFIG        = Path("../python_mbconv/configs/model_config_mbconv_micro_merged.json")
SRC_CONFIG        = Path("../python_mbconv/configs/model_config_mbconv_micro.json")

# Какие клипы есть в старом датасете
OLD_CLIPS = [f"pt{i}" for i in range(1, 16)]  # pt1–pt15

# Какие клипы в новых видео
NEW_CLIPS = [f"pt{i}" for i in range(16, 22)]  # pt16–pt21

# BPE параметры
VOCAB_SIZE = 1200
BOS_TOKEN  = "<|bos|>"
EOS_TOKEN  = "<|eos|>"
PAD_TOKEN  = "<|pad|>"
UNK_TOKEN  = "<|unk|>"

# Функция очистки текста
def clean_text(text):
    """Убрать пунктуацию, lowercase, оставить апострофы."""
    return re.sub(r"[^\w\s']", '', text.lower()).strip()

print(f"Старый датасет:   {OLD_DATA_PATH}")
print(f"Новые видео:      {NEW_RAW_PATH}")
print(f"Итоговый датасет: {MERGED_DATA_PATH}")

Старый датасет:   D:\Datasets\Dataset_720_proc
Новые видео:      D:\Datasets\Dataset_VSR
Итоговый датасет: D:\Datasets\Dataset_merged_bpe


## Шаг 1. Обработка новых видео → pkl

Используем `LipReading2Preprocessor` из `find_face.py` для извлечения
области губ из каждого кадра и сохранения в `.pkl` формате.

In [2]:
# %pip install --upgrade opencv-python
# %pip install "numpy<2"
# %pip install dlib
# %pip install --force-reinstall dlib
# %pip install dlib==19.24.2
# %pip install dlib==19.24.6 --find-links https://github.com/z-mahmud22/Dlib_Windows_Python3.x/releases/tag/v19.24.6
# %pip install https://github.com/z-mahmud22/Dlib_Windows_Python3.x/releases/download/v19.24.6/dlib-19.24.6-cp311-cp311-win_amd64.whl


import sys
sys.path.insert(0, ".")

from find_face import LipReading2Preprocessor

preprocessor = LipReading2Preprocessor(
    raw_data_path=str(NEW_RAW_PATH),
    processed_data_path=str(NEW_PROCESSED_PATH),
)

print(f"Обработка видео из {NEW_RAW_PATH} ...")
preprocessor.process_videos()
print("Готово.")

ImportError: DLL load failed while importing _dlib_pybind11: A dynamic link library (DLL) initialization routine failed.

In [ ]:
# Проверка: сколько pkl файлов создано
import pickle

new_pkl_count = 0
for clip in NEW_CLIPS:
    clip_dir = NEW_PROCESSED_PATH / clip
    if clip_dir.exists():
        files = list(clip_dir.glob("*.pkl"))
        new_pkl_count += len(files)
        print(f"  {clip}: {len(files)} файлов")
    else:
        print(f"  {clip}: папка не найдена")

print(f"\nВсего новых pkl: {new_pkl_count}")

# Проверяем формат первого файла
sample_pkl = next(NEW_PROCESSED_PATH.glob("**/*.pkl"))
with open(sample_pkl, "rb") as f:
    sample = pickle.load(f)
print(f"\nПример: {sample_pkl.name}")
print(f"  keys: {list(sample.keys())}")
print(f"  num_frames: {sample['num_frames']}")
print(f"  input_ids (GPT-2): {sample['input_ids'][:10]}...")
print(f"  frame shape: {sample['frames'][0].shape}")

## Шаг 2. Сбор объединённого текстового корпуса

Тексты из двух источников:
- Старый датасет: декодируем `input_ids` через GPT-2 → очищаем
- Новый датасет: декодируем `input_ids` через GPT-2 → очищаем

In [ ]:
from transformers import AutoTokenizer

gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")

corpus_texts = []   # все тексты для обучения BPE
all_files    = []   # (pkl_path, source, cleaned_text) для шага 4

# ── Старый датасет ──
old_count = 0
for clip_name in sorted(OLD_CLIPS):
    clip_dir = OLD_DATA_PATH / clip_name
    if not clip_dir.exists():
        continue
    for pkl_path in sorted(clip_dir.glob("*.pkl"),
                           key=lambda f: int(f.stem.split("_")[-1])):
        with open(pkl_path, "rb") as f:
            data = pickle.load(f)
        raw = gpt2_tokenizer.decode(data["input_ids"])
        text = clean_text(raw)
        if text:
            corpus_texts.append(text)
            all_files.append((pkl_path, "old", text))
            old_count += 1

# ── Новый датасет ──
new_count = 0
for clip_name in sorted(NEW_CLIPS):
    clip_dir = NEW_PROCESSED_PATH / clip_name
    if not clip_dir.exists():
        continue
    for pkl_path in sorted(clip_dir.glob("*.pkl"),
                           key=lambda f: int(f.stem.split("_")[-1])):
        with open(pkl_path, "rb") as f:
            data = pickle.load(f)
        raw = gpt2_tokenizer.decode(data["input_ids"])
        text = clean_text(raw)
        if text:
            corpus_texts.append(text)
            all_files.append((pkl_path, "new", text))
            new_count += 1

print(f"Файлов из старого датасета: {old_count}")
print(f"Файлов из нового датасета:  {new_count}")
print(f"Всего текстов для корпуса:  {len(corpus_texts)}")
print()
print("=== Примеры (старые) ===")
for t in corpus_texts[:3]:
    print(f"  {repr(t)}")
print("=== Примеры (новые) ===")
for t in corpus_texts[old_count:old_count+3]:
    print(f"  {repr(t)}")

## Шаг 3. Анализ объединённого корпуса

In [ ]:
from collections import Counter

all_text = " ".join(corpus_texts)
words    = re.findall(r"\b\w+\b", all_text)
word_freq = Counter(words)

print(f"Всего символов в корпусе:   {len(all_text):,}")
print(f"Всего слов (с повторами):   {len(words):,}")
print(f"Уникальных слов:            {len(word_freq):,}")
print()
print("Топ-20 слов:")
for word, cnt in word_freq.most_common(20):
    print(f"  {word!r:20s} {cnt}")

## Шаг 4. Обучение BPE-токенизатора на объединённом корпусе

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(BPE(unk_token=UNK_TOKEN))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=[PAD_TOKEN, BOS_TOKEN, EOS_TOKEN, UNK_TOKEN],
    min_frequency=1,
    show_progress=True,
)

tokenizer.train_from_iterator(corpus_texts, trainer=trainer)

PAD_TOKEN_ID = tokenizer.token_to_id(PAD_TOKEN)
BOS_TOKEN_ID = tokenizer.token_to_id(BOS_TOKEN)
EOS_TOKEN_ID = tokenizer.token_to_id(EOS_TOKEN)
actual_vocab = tokenizer.get_vocab_size()

print(f"Реальный vocab_size: {actual_vocab}")
print(f"PAD={PAD_TOKEN_ID}, BOS={BOS_TOKEN_ID}, EOS={EOS_TOKEN_ID}")

# Сохраняем
TOKENIZER_PATH.mkdir(parents=True, exist_ok=True)
tokenizer.save(str(TOKENIZER_PATH / "tokenizer.json"))
print(f"Сохранён: {TOKENIZER_PATH / 'tokenizer.json'}")

# Проверка
for t in corpus_texts[:3]:
    enc = tokenizer.encode(t)
    print(f"  {repr(t[:50])} → {enc.ids}")

## Шаг 5. Сохранение объединённого датасета с новыми input_ids

Все pkl (старые + новые) копируются в `MERGED_DATA_PATH`
с ретокенизированными `input_ids`.

In [ ]:
converted = 0
skipped   = 0

for pkl_src, source, text in all_files:
    # Определяем имя клипа (ptN) из пути
    clip_name = pkl_src.parent.name   # e.g. "pt1", "pt16"
    file_name = pkl_src.name          # e.g. "pt1_0.pkl"

    pkl_dst = MERGED_DATA_PATH / clip_name / file_name
    pkl_dst.parent.mkdir(parents=True, exist_ok=True)

    with open(pkl_src, "rb") as f:
        data = pickle.load(f)

    new_ids = tokenizer.encode(text).ids
    if len(new_ids) == 0:
        print(f"[!] Пустая токенизация: {pkl_src.name}, пропуск")
        skipped += 1
        continue

    data["input_ids"] = new_ids

    with open(pkl_dst, "wb") as f:
        pickle.dump(data, f)

    converted += 1

print(f"Конвертировано: {converted} файлов")
print(f"Пропущено:      {skipped} файлов")
print(f"Датасет:        {MERGED_DATA_PATH}")
print()

# Статистика по клипам
for clip_dir in sorted(MERGED_DATA_PATH.iterdir()):
    if clip_dir.is_dir():
        n = len(list(clip_dir.glob("*.pkl")))
        print(f"  {clip_dir.name}: {n} файлов")

## Шаг 6. Обновление конфига модели

In [ ]:
import json

with open(SRC_CONFIG, "r", encoding="utf-8") as f:
    config = json.load(f)

old_vocab = config["vocab_size"]
config["vocab_size"] = actual_vocab

with open(DST_CONFIG, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f"Конфиг: {DST_CONFIG}")
print(f"  vocab_size: {old_vocab} → {actual_vocab}")
print()
print(json.dumps(config, indent=2, ensure_ascii=False))

## Шаг 7. Проверка объединённого датасета

In [ ]:
import numpy as np

# Статистика длин токенов
token_lengths = []
for pkl_path, _, text in all_files:
    token_lengths.append(len(tokenizer.encode(text).ids))

print(f"Длина последовательности токенов:")
print(f"  min:    {min(token_lengths)}")
print(f"  max:    {max(token_lengths)}")
print(f"  median: {int(np.median(token_lengths))}")
print(f"  mean:   {np.mean(token_lengths):.1f}")
print()

# Проверка нескольких файлов
print("=== Примеры из объединённого датасета ===")
for clip in ["pt1", "pt16"]:
    clip_dir = MERGED_DATA_PATH / clip
    if not clip_dir.exists():
        continue
    sample_path = sorted(clip_dir.glob("*.pkl"))[0]
    with open(sample_path, "rb") as f:
        s = pickle.load(f)
    decoded = tokenizer.decode(s["input_ids"])
    print(f"  {clip}/{sample_path.name}: frames={s['num_frames']}, "
          f"ids={s['input_ids'][:8]}..., text={repr(decoded[:60])}")

## Итог

После выполнения:
- Объединённый датасет: `D:/Datasets/Dataset_merged_bpe/` (pt1–pt21)
- Токенизатор: `configs/bpe_tokenizer_merged/tokenizer.json`
- Конфиг: `python_mbconv/configs/model_config_mbconv_micro_merged.json`

**Для обучения в `train_bpe.ipynb` изменить:**
```python
DATA_PATH         = "D:/Datasets/Dataset_merged_bpe"
TOKENIZER_PATH    = "../python/configs/bpe_tokenizer_merged/tokenizer.json"
MODEL_CONFIG_PATH = "configs/model_config_mbconv_micro_merged.json"

# Обновить сплиты:
TRAIN_CLIPS = [f"pt{i}" for i in range(1, 18)]   # pt1–pt17 (расширенный train)
VAL_CLIPS   = [f"pt{i}" for i in range(18, 20)]   # pt18–pt19
TEST_CLIPS  = [f"pt{i}" for i in range(20, 22)]   # pt20–pt21
```